In [1]:
import sys
import os
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../../..')))

load_dotenv()


True

In [2]:
SYSTEM_PROMPT = """你是一个会使用工具的 ReAct Agent。请严格按照下面的格式循环工作：

Thought: 用一句话写出你接下来要做什么、为什么。
Action: 工具名[参数]
Observation: （这一行不要你写，由系统填充工具执行结果）

可以使用的工具：
- get_weather[城市名]：查询指定城市当前天气，返回温度（摄氏度）和天气描述。
- calculate[数学表达式]：计算一个数学表达式的值，例如 calculate[32 - 30]。
- finish[最终答案]：当你已经能回答用户问题时，用这个动作结束。

规则：
1. 每一步只输出一个 Thought + 一个 Action，输出 Action 后立刻停下，不要自己编 Observation。
2. Action 名字必须是上面列出的三个之一，参数必须放在方括号里。
3. 拿到 Observation 后再继续下一轮 Thought + Action。
4. 一旦信息已经够回答用户问题，立刻用 finish[答案] 结束。

下面是一个示例：

问题：上海今天比北京热多少度？
Thought: 我需要分别查上海和北京的气温，再算差值。先查上海。
Action: get_weather[上海]
Observation: 多云, 28°C, 湿度 75%
Thought: 上海 28°C，再查北京。
Action: get_weather[北京]
Observation: 晴, 24°C, 湿度 40%
Thought: 上海 28，北京 24，算差。
Action: calculate[28 - 24]
Observation: 4
Thought: 信息齐了，可以回答了。
Action: finish[上海今天比北京热 4°C]
"""

In [ ]:
import re
from app.llm import client
from app.config import DEFAULT_MODEL

# 1 工具实现（mock 版，演示用）
def get_weather(city: str) -> str:
    fake_db = {"北京": "晴, 32°C, 湿度 35%", "上海": "多云, 28°C, 湿度 75%"}
    return fake_db.get(city, f"未找到 {city} 的天气数据")

def calculate(expression: str) -> str:
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"

TOOLS = {"get_weather": get_weather, "calculate": calculate}

# 2 解析模型输出里的 Action 行
ACTION_RE = re.compile(r"Action:\s*(\w+)\[(.*?)\]", re.DOTALL)

def parse_action(text: str):
    m = ACTION_RE.search(text)
    if not m:
        return None, None
    return m.group(1).strip(), m.group(2).strip()

# 3 主循环
def react(question: str, max_steps: int = 6, verbose: bool = True) -> str:
    history = ""
    for step in range(max_steps):
        prompt = f"问题：{question}\n" if step == 0 else f"问题：{question}\n{history}"

        resp = client.chat.completions.create(
            model=DEFAULT_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ],
            stop=["Observation:"],   # ⚠️ 关键：在这里截断
            temperature=0,
        )
        output = resp.choices[0].message.content.strip()
        if verbose:
            print(f"\n--- Step {step+1} ---\n{output}")

        action, arg = parse_action(output)
        if action is None:
            return f"⚠️ 模型没输出合法 Action，原文：\n{output}"
        if action == "finish":
            return arg

        obs = TOOLS[action](arg) if action in TOOLS else f"Error: 未注册工具 {action}"

        if verbose:
            print(f"Observation: {obs}")
        history += f"\n{output}\nObservation: {obs}"

    return "⚠️ 达到 max_steps 仍未给出 finish"

In [5]:
react("北京和上海今天气温差多少？北京天气如果超过 30°C 就给我推荐一杯冷饮。")


--- Step 1 ---
Thought: 先查询北京的气温。  
Action: get_weather[北京]
Observation: 晴, 32°C, 湿度 35%

--- Step 2 ---
Thought: 需要查询上海的气温以计算温差。  
Action: get_weather[上海]
Observation: 多云, 28°C, 湿度 75%

--- Step 3 ---
Thought: 计算北京与上海的气温差。  
Action: calculate[32 - 28]
Observation: 4

--- Step 4 ---
Thought: 北京温度32°C已超过30°C，需推荐冷饮并整合温差结果。  
Action: finish[北京和上海今天气温差4°C。北京天气超过30°C，推荐你喝冰镇柠檬水]


'北京和上海今天气温差4°C。北京天气超过30°C，推荐你喝冰镇柠檬水'